# Lesson 06 · Universal Properties

Five lessons of preparation pay off: we now define objects by how everything else maps to them — and the definitions pin their subjects down up to unique isomorphism.

This notebook collects every Python block from the optional labs and exercises of [the lesson page](../lesson-06-universal-properties.html), in page order. Run the cells from top to bottom; later cells use names defined in earlier ones.

Python is optional in this course — the lesson is complete without it. *Generated by `tools/sync_code.py`: edit the lesson page, not this file.*

## §6 — Universal properties by hand

**`setup.py`** — Lesson 1’s category class, carried over

In [1]:
from itertools import product

class FiniteCategory:
    """objects, arrows (name -> (dom, cod)), a composition table,
       and a chosen identity arrow for each object."""
    def __init__(self, objects, arrows, comp, ident):
        self.obj, self.ar = set(objects), arrows
        self.comp, self.id = comp, ident

    def dom(self, a): return self.ar[a][0]
    def cod(self, a): return self.ar[a][1]
    def composable(self, g, f): return self.dom(g) == self.cod(f)

    def check(self):
        # 1. each identity is typed X -> X
        for X in self.obj:
            assert self.ar[self.id[X]] == (X, X)
        # 2. every composable pair has a composite, correctly typed
        for g, f in product(self.ar, repeat=2):
            if self.composable(g, f):
                gf = self.comp[(g, f)]
                assert self.ar[gf] == (self.dom(f), self.cod(g))
        # 3. identity law on both sides
        for f in self.ar:
            X, Y = self.ar[f]
            assert self.comp[(self.id[Y], f)] == f
            assert self.comp[(f, self.id[X])] == f
        # 4. associativity
        for h, g, f in product(self.ar, repeat=3):
            if self.composable(h, g) and self.composable(g, f):
                assert self.comp[(h, self.comp[(g, f)])] == \
                       self.comp[(self.comp[(h, g)], f)]
        return True

**`universal.py`** — finders for universal objects

In [2]:
def arrows_between(C, X, Y):
    return [a for a, (d, c) in C.ar.items() if d == X and c == Y]

def terminal_objects(C):
    "T is terminal iff every object has EXACTLY ONE arrow into T."
    return [T for T in C.obj
            if all(len(arrows_between(C, X, T)) == 1 for X in C.obj)]

def initial_objects(C):
    return [I for I in C.obj
            if all(len(arrows_between(C, I, X)) == 1 for X in C.obj)]

def is_product(C, P, pA, pB):
    "Check the universal property of (P, pA: P->A, pB: P->B)."
    A, B = C.cod(pA), C.cod(pB)
    for X in C.obj:
        for f in arrows_between(C, X, A):
            for g in arrows_between(C, X, B):
                mediators = [m for m in arrows_between(C, X, P)
                             if C.comp[(pA, m)] == f and C.comp[(pB, m)] == g]
                if len(mediators) != 1:      # exists AND is unique
                    return False
    return True

def find_products(C, A, B):
    return [(P, pA, pB) for P in C.obj
            for pA in arrows_between(C, P, A)
            for pB in arrows_between(C, P, B)
            if is_product(C, P, pA, pB)]

# The divisibility order on {1, 2, 3, 6}:  arrow a -> b  iff  a divides b
objs = {1, 2, 3, 6}
ar   = {f"{a}|{b}": (a, b) for a in objs for b in objs if b % a == 0}
comp = {}
for g, (gs, gt) in ar.items():
    for f, (fs, ft) in ar.items():
        if ft == gs:
            comp[(g, f)] = f"{fs}|{gt}"      # thin category: forced
ident = {n: f"{n}|{n}" for n in objs}
Div = FiniteCategory(objs, ar, comp, ident)

print("initial: ", initial_objects(Div))
print("terminal:", terminal_objects(Div))
print("2 x 3 =", find_products(Div, 2, 3))   # expect gcd = 1
print("2 x 6 =", find_products(Div, 2, 6))   # expect gcd = 2

initial:  [1]
terminal: [6]
2 x 3 = [(1, '1|2', '1|3')]
2 x 6 = [(2, '2|2', '2|6')]


**`no_products.py`** — a category with no products

In [3]:
# Z3 = one object '*', arrows {0,1,2}, composition = addition mod 3
zar  = {n: ('*', '*') for n in range(3)}
zcmp = {(a, b): (a + b) % 3 for a in range(3) for b in range(3)}
Z3   = FiniteCategory({'*'}, zar, zcmp, {'*': 0})

print(find_products(Z3, '*', '*'))    # exhaustive search over all
                                      # candidate (P, p, q) triples

[]


**`pairing.py`** — the product in Set

In [4]:
def pairing(f, g):
    "the unique mediating map  <f, g> : X -> A x B"
    return lambda x: (f(x), g(x))

fst = lambda p: p[0]
snd = lambda p: p[1]

X = [0, 1, 2]
f = lambda x: x + 10          # f : X -> A
g = lambda x: x * x           # g : X -> B
m = pairing(f, g)

# both triangles of the universal property commute:
assert all(fst(m(x)) == f(x) for x in X)
assert all(snd(m(x)) == g(x) for x in X)
print([m(x) for x in X])

[(10, 0), (11, 1), (12, 4)]


## Exercise 8

**`coproducts.py`**

In [5]:
def find_coproducts(C, A, B):
    hits = []
    for P in C.obj:
        for iA in arrows_between(C, A, P):
            for iB in arrows_between(C, B, P):
                ok = True
                for X in C.obj:
                    for f in arrows_between(C, A, X):
                        for g in arrows_between(C, B, X):
                            med = [m for m in arrows_between(C, P, X)
                                   if C.comp[(m, iA)] == f
                                   and C.comp[(m, iB)] == g]
                            if len(med) != 1: ok = False
                if ok: hits.append((P, iA, iB))
    return hits

print(find_coproducts(Div, 2, 3))   # [(6, '2|6', '3|6')]  -- the lcm
print(find_coproducts(Z3, '*', '*'))  # []

[(6, '2|6', '3|6')]
[]
